# P14-P15 · Cambios de red y oferta entre publicaciones GTFS

> **¿Qué cambió entre 2024 y 2026 y cómo varió la oferta programada por hora?**

Últimas publicaciones GTFS de 2024 y 2025, más el feed vigente del 14 de noviembre de 2026.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/evolucion_gtfs/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-historicos-2024 gtfs-historicos-2025 gtfs-red-movilidad


## 1 · El problema

Comparar publicaciones no es restar sus conteos. Una ruta puede conservar el nombre y
cambiar de trazado; un identificador puede cambiar sin que la realidad cambie; una
versión puede modelar horarios absolutos donde otra usa frecuencias. P14 pregunta qué
cambió y P15 cuánto cambió la oferta: ambas requieren fijar identidad, grano y método.

La ejecución usa tres publicaciones oficiales reales y conserva sus fechas. Es una
trayectoria reciente 2024–2026, no una supuesta serie completa de trece años.


## 2 · La solución que se va a aplicar

Se leen las últimas publicaciones de 2024 y 2025 desde ZIP anuales y la vigente de 2026.
Se inventaría cada tabla, se comparan conjuntos de rutas y paradas de embarque, y se
firman las secuencias de coordenadas para detectar cambios geométricos. La oferta se
recalcula en las tres versiones con la misma expansión y el mismo `service_id` laboral.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.evolucion_gtfs import solution as evolucion

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Nombrar las publicaciones por fecha


La etiqueta temporal pertenece al dato, no al momento de análisis.

> ⚠️ Llamar a ambos archivos 'GTFS' borra qué estado de la red representa cada uno.


In [ ]:
[(r.etiqueta, r.ruta.name, r.miembro or 'ZIP directo') for r in evolucion.REFERENCIAS]


### Paso 2 · Inventariar antes de comparar


Volúmenes y esquemas muestran si las cargas son comparables.

> ⚠️ El salto de viajes declarados no se interpreta antes de revisar la mecánica de frecuencias.


In [ ]:
resultados = evolucion.analizar()
resultados['inventario']


### Paso 3 · Comparar rutas como conjuntos


Se separan permanencias, altas y bajas.

> ⚠️ Una baja de `route_id` no prueba que el recorrido físico dejó de existir.


In [ ]:
resultados['identidades'].query("entidad == 'rutas'")


### Paso 4 · Comparar solo paradas de embarque


El denominador excluye estaciones, accesos y nodos de pasillo.

> ⚠️ Comparar todo `stops.txt` confunde cambios de jerarquía con paradas nuevas.


In [ ]:
resultados['identidades'].query("entidad == 'paradas de embarque'")


### Paso 5 · Firmar el trazado, no solo el ID


Cada geometría se resume desde su secuencia completa de puntos.

> ⚠️ Un `shape_id` estable puede ocultar coordenadas nuevas; uno nuevo puede dibujar lo mismo.


In [ ]:
resultados['trazados'].query('cambio_de_conjunto').head(15)


### Paso 6 · Recalcular la oferta por hora


La misma función combina frecuencias y horarios absolutos en ambas versiones.

> ⚠️ Comparar filas de `trips.txt` daría una serie rota: una ventana representa muchas salidas.


In [ ]:
resultados['oferta_por_hora'].head(30)


### Paso 7 · Medir el cambio total


Se publica diferencia absoluta y porcentual con su período.

**En Excel:** `=(B3/B2)-1`

Debe dar: **−0,76 % y −5,88 %**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Tres puntos describen una trayectoria reciente, no toda la serie 2014–2026.


In [ ]:
resultados['oferta_total']


### Paso 8 · Dejar preparada la ampliación


Agregar un año es agregar una FeedRef; la definición no cambia.

> ⚠️ Completar años con cifras copiadas de informes rompería la comparabilidad del método.


In [ ]:
{'publicaciones_ejecutadas': len(evolucion.REFERENCIAS), 'primera': evolucion.REFERENCIAS[0].etiqueta, 'ultima': evolucion.REFERENCIAS[-1].etiqueta}


## 4 · El resultado

La oferta laboral calculada con idéntica definición pasa de 68 533 salidas en 2024 a 68 011 en 2025 y 64 011 en 2026; el último cambio es −5,88 %.

## 5 · Qué queda aprendido

1. La identidad en el tiempo no se supone.
2. El trazado se compara por geometría.
3. La serie debe recalcular cada versión con la misma definición.
4. Tres puntos describen una trayectoria reciente, no una historia completa.
